# Placement 1 — SAE, Jacobian Lens, and activation patching

FOSE7901 research notebook for the report. It runs the three methods already implemented in `multilingual_mechinterp` on one shared set of English–Persian culture questions, then writes figures and a research record under `output/placement_1/`.

**Research question.** How do sparse autoencoders, the Jacobian Lens, and activation patching differ in the internal representations and causal mechanisms they reveal from the same controlled prompts?

| Report section | Notebook section |
| --- | --- |
| Introduction and background | Stage 1 |
| Experimental setup and prompts | Setup, prompt set, baseline behaviour |
| SAE | Stage 2 |
| Jacobian Lens | Stage 3 |
| Activation patching | Stage 4 |
| Comparison | Stage 6 |
| Reproducibility appendix | Stage 7 |

Run the notebook from top to bottom on the machine that has Gemma 2 9B (see `docs/others/runpod_s3_models.md`). Edit the control cell before the long stages. Outputs land in `output/placement_1/records/` and `output/placement_1/figures/`. The last code cell writes `report_notes.md`, which is the draft you can lift into the report.

Jacobian fitting is the slow stage: each fit prompt needs one backward pass per `d_model / JLENS_DIM_BATCH` chunk, for every source layer. Patching then runs one forward per layer per item.


## What is already implemented

The methods are in the package. This notebook calls them and records a placement-sized experiment.

| Piece | Code | Already exercised |
| --- | --- | --- |
| Tied SAE (train, FVU, L0, feature ranks) | `src/multilingual_mechinterp/sae/` | `output/gemma_9B/sae/` from `Full_mechanistic_interp_gemma_9b.ipynb` |
| Jacobian fit and layer-wise readouts | `src/multilingual_mechinterp/jlens/` | `output/gemma_9B/jlens/` |
| Residual activation patching | `src/multilingual_mechinterp/patching/` | `output/gemma_9B/patching/` and `comparison/` |
| Curated EN / FA / BN / UR questions | `data/all200questions_persianMiddleEastCulture.json` | 200 items |

The earlier Gemma notebook is the development pipeline. This notebook is the report run. Two placement requirements are handled here in particular:

- Activation patching replaces **one layer at a time** (`window=1`). The package default replaces every layer from the start layer through the end, which is why the earlier curves often peak at layer 0.
- SAE feature ablation and a Jacobian-lens concept-direction steer are run on a showcase item and scored as MCQ margin changes.

An earlier autointerp file may list strongly activating features, including feature 516. A high activation is a description of the encoder. It becomes evidence about meaning only when the contexts and, where you run it, the intervention agree.


## Setup

On the pod, install the wheel first (`docs/others/runpod_setup.md`), then run this notebook from the repo. The next code cell loads `notebooks/runpod_setup.py`, which points `MODEL_NAME` at `/models/gemma-2-9b`. Leave that path as it is.


In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    print("Drive mounted:", Path("/content/drive/MyDrive").exists())
except ImportError:
    print("Not Colab — skip mount.")


In [ ]:
from pathlib import Path
import os
import runpy


def _resolve_setup_script() -> Path:
    try:
        here = Path.cwd()
    except FileNotFoundError:
        for fallback in ("/workspace/multilingual-mechinterp", "/workspace", "/root", "/"):
            if Path(fallback).is_dir():
                os.chdir(fallback)
                break
        here = Path.cwd()
    candidates = [
        here / "runpod_setup.py",
        here / "notebooks" / "runpod_setup.py",
        here.parent / "notebooks" / "runpod_setup.py",
        Path("/workspace/multilingual-mechinterp/notebooks/runpod_setup.py"),
        Path("/content/drive/MyDrive/multilingual_mech/notebooks/runpod_setup.py"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "runpod_setup.py not found. On RunPod, start the kernel in the repo or in notebooks/."
    )


_setup = runpy.run_path(str(_resolve_setup_script()))
globals().update({k: _setup[k] for k in _setup["EXPORTS"]})

import csv
import json
import platform
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import torch

import multilingual_mechinterp
from multilingual_mechinterp.data import (
    load_culture_questions,
    option_token_ids,
    score_mcq_prompt,
    score_options_from_logits,
)
from multilingual_mechinterp.jlens import fit, from_hf, intervene_jlens, run_jlens
from multilingual_mechinterp.models.hooks import hooked_forward
from multilingual_mechinterp.models.loader import LoadedModel
from multilingual_mechinterp.patching import collect_activations, get_blocks, patch_residuals, run_patching
from multilingual_mechinterp.sae import (
    ablate_features,
    analyze_features,
    extract_residual_activations,
    steer_features,
    sweep_l1,
)
from multilingual_mechinterp.utils import ensure_dir, load_config, save_json

cfg_path = CONFIG_DIR / "gemma2_9b.yaml"
cfg = load_config(cfg_path) if cfg_path.exists() else {}

PLACE = ensure_dir(OUTPUT_DIR / "placement_1")
REC = ensure_dir(PLACE / "records")
FIG = ensure_dir(PLACE / "figures")
SAE_DIR = ensure_dir(PLACE / "sae")

plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "font.size": 11})
print("ROOT  ", ROOT)
print("OUTPUT", PLACE)
print("MODEL ", MODEL_NAME)


## Run controls

Change these before loading the model. `N_ITEMS = 8` is a report-sized sample of the 200 curated questions. The plan allows 30–100 pairs and also allows a smaller sample when compute is tight; the prompt-set cell records that justification.

`PATCH_WINDOW = 1` replaces a single layer. Leave it at 1 for the placement plan.

Set `RUN_JLENS` or `RUN_PATCHING` to `False` only when you are re-running an earlier stage. The comparison cell skips whatever did not run.


In [ ]:
# How many curated questions to share across all three methods.
N_ITEMS = 8
# None uses the first N_ITEMS. Or pin ids, e.g. [1, 2, 3, 4, 5, 6, 7, 8].
ITEM_IDS = None

SAE_LAYER = None  # None → configs/gemma2_9b.yaml sae.layer, else middle layer
SAE_MAX_TOKENS = 2048
SAE_EPOCHS = 4
SAE_ALPHAS = [3.0e-4, 8.6e-4, 3.0e-3]
SAE_BATCH = 256
SAE_RATIO = 2.0
SAE_DEVICE = None  # None → same device as the model. Use "cpu" if the SAE step runs out of GPU memory.
SAE_TOP_K = 10

JLENS_N_SOURCE_LAYERS = 4
JLENS_FIT_PROMPTS = 2
JLENS_DIM_BATCH = 16
JLENS_SKIP_FIRST = 16
JLENS_TOP_K = 8

PATCH_WINDOW = 1
STEER_RESIDUAL_SCALE = 0.5
MAX_LENGTH = 512
SEED = 0

RUN_SAE = True
RUN_JLENS = True
RUN_PATCHING = True
RUN_INTERVENTIONS = True

SHOWCASE_INDEX = 0
INFERENCE_MARKERS = ("symbol", "refer", "gesture", "custom", "relationship")

torch.manual_seed(SEED)
STATUS = {}
STAGE_TIMES = {}
SAE = None
SAE_LAYER_USED = None
LENS = None
JLENS_LAYERS = []
PAIRS = []
BASELINE_ROWS = []
SAE_FEATURE_ROWS = []
SAE_INTERVENTION = None
JLENS_ROWS = []
JLENS_INTERVENTION = None
PATCH_ROWS = []


In [ ]:
if USE_TINY_OFFLINE:
    raise RuntimeError(
        "Placement 1 needs the real model. Set USE_TINY_OFFLINE = False in notebooks/runpod_setup.py."
    )

print("Loading", MODEL_NAME)
model = load_experiment_model(MODEL_NAME)
if not isinstance(model, LoadedModel):
    raise TypeError(f"Expected LoadedModel, got {type(model).__name__}")

param = next(model.model.parameters())
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"


def _git(args):
    try:
        return subprocess.check_output(args, cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


setup_record = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "research_question": (
        "How do sparse autoencoders, the Jacobian Lens, and activation patching differ "
        "in the internal representations and causal mechanisms they reveal from the same "
        "controlled LLM prompts?"
    ),
    "model_name": model.name,
    "n_layers": model.n_layers,
    "d_model": int(param.shape[-1]) if param.ndim >= 1 else None,
    "dtype": str(param.dtype),
    "device": str(model.device),
    "gpu": gpu_name,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "package": getattr(multilingual_mechinterp, "__version__", None),
    "git_commit": _git(["git", "rev-parse", "HEAD"]),
    "git_branch": _git(["git", "rev-parse", "--abbrev-ref", "HEAD"]),
    "config": str(cfg_path) if cfg_path.exists() else None,
    "controls": {
        "N_ITEMS": N_ITEMS,
        "ITEM_IDS": ITEM_IDS,
        "SAE_ALPHAS": SAE_ALPHAS,
        "SAE_EPOCHS": SAE_EPOCHS,
        "JLENS_N_SOURCE_LAYERS": JLENS_N_SOURCE_LAYERS,
        "JLENS_FIT_PROMPTS": JLENS_FIT_PROMPTS,
        "JLENS_DIM_BATCH": JLENS_DIM_BATCH,
        "PATCH_WINDOW": PATCH_WINDOW,
    },
}
save_json(setup_record, REC / "experiment_setup.json")
print(
    f"Ready: {model.n_layers} layers, dtype={param.dtype}, device={model.device}, gpu={gpu_name}"
)


In [ ]:
def dump(name, obj):
    path = save_json(obj, REC / f"{name}.json")
    print("wrote", path)
    return path


def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("saved", path)
    return str(path)


class timed:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        print(f"--- {self.name} ---")
        return self

    def __exit__(self, exc_type, exc, tb):
        dt = time.perf_counter() - self.t0
        STAGE_TIMES[self.name] = round(dt, 2)
        save_json(STAGE_TIMES, REC / "stage_times.json")
        label = "failed" if exc_type else "ok"
        print(f"--- {self.name} {label} in {dt:.1f}s ---")
        return False


def encode_prompt(prompt):
    tok = model.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    tok = {k: v.to(model.device) for k, v in tok.items()}
    mask = tok.get("attention_mask")
    if mask is None:
        pos = int(tok["input_ids"].shape[-1]) - 1
    else:
        pos = int(mask[0].sum().item()) - 1
    return tok, pos


def last_residual(prompt, layer):
    tok, pos = encode_prompt(prompt)
    with torch.no_grad(), hooked_forward(model.model, layers=[int(layer)]) as cache:
        model.model(**tok)
        acts = cache.get(f"blocks.{int(layer)}")
    return acts[0, pos].detach().float().cpu(), pos


def logits_with_residual(prompt, layer, residual, pos):
    tok, pos = encode_prompt(prompt)
    dtype = next(model.model.parameters()).dtype
    values = residual.detach().to(device=model.device, dtype=dtype).unsqueeze(0)
    positions = torch.tensor([pos], device=model.device)
    with torch.no_grad(), patch_residuals(model.model, {int(layer): values}, positions):
        out = model.model(**tok)
    return out.logits[0, pos].detach().float().cpu()


def slim_score(score):
    return {
        "pred_letter": score["pred_letter"],
        "correct": bool(score["correct"]),
        "p_correct": float(score["p_correct"]),
        "margin": float(score["margin"]),
        "probs": {k: float(v) for k, v in score["probs"].items()},
    }


def score_prompt(prompt, letter):
    return slim_score(score_mcq_prompt(model, prompt, correct_letter=letter))


def score_logits(logits_row, letter):
    opt_ids = option_token_ids(model.tokenizer)
    return slim_score(score_options_from_logits(logits_row, opt_ids, correct_letter=letter))


def jaccard(left, right):
    a, b = set(left), set(right)
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


def empty_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("Helpers ready.")


## Stage 1 — Baseline method notes

This cell records the technical foundation for the report: which component each method reads, what the code does, and where this run departs from a full paper replication. It is a method note, in line with the plan's request for a short technical foundation.

| Method | Baseline | What this run uses |
| --- | --- | --- |
| SAE | Huben et al. (2024), sparse features in residual activations | Tied SAE in `sae/learned_dict.py` (dictionary rows tied to the encoder), trained on this run's residual tokens |
| Jacobian Lens | Gurnee et al. (2026), Jacobian transport of intermediate residuals into vocabulary space | `jlens/fitting.py` averages per-prompt Jacobians on the last few source layers, then `run_jlens` unembeds the transported residual |
| Activation patching | Dumas et al. (2025), patch a source run into a target run | `run_patching(..., window=1)` writes the English residual into one Persian-run layer and rescores the MCQ margin |


In [ ]:
method_notes = {
    "sae": {
        "baseline": "Huben et al. (2024), Sparse Autoencoders Find Highly Interpretable Features in Language Models",
        "code": [
            "src/multilingual_mechinterp/sae/learned_dict.py",
            "src/multilingual_mechinterp/sae/train.py",
            "src/multilingual_mechinterp/sae/features.py",
            "src/multilingual_mechinterp/sae/intervention.py",
        ],
        "component": "Residual stream after the chosen transformer block (last non-padding token for feature reads; all non-padding tokens for training).",
        "procedure": "Extract residuals, train a tied SAE across a short list of L1 coefficients, rank features by activation, then ablate or steer one feature and write the decoded residual back.",
        "departures": [
            "The SAE is trained for this run on the placement prompts. A published Gemma Scope dictionary is not loaded.",
            "Expansion ratio is 2 and the token sample is SAE_MAX_TOKENS, so the dictionary is a placement instrument, not a full-scale feature dictionary.",
            "Feature ids are unlabeled. Semantic names would require a separate interpreter pass.",
        ],
    },
    "jlens": {
        "baseline": "Gurnee et al. (2026), Verbalizable Representations Form a Global Workspace in Language Models",
        "code": [
            "src/multilingual_mechinterp/jlens/fitting.py",
            "src/multilingual_mechinterp/jlens/apply.py",
            "src/multilingual_mechinterp/jlens/intervention.py",
        ],
        "component": "Residual stream at a few late source layers, transported toward the final residual and then unembedded.",
        "procedure": "Fit Jacobians on JLENS_FIT_PROMPTS prompts, rank vocabulary tokens at each source layer, and add or subtract the output-embedding direction of one top token.",
        "departures": [
            "The fit uses a handful of prompts and the last JLENS_N_SOURCE_LAYERS source layers.",
            "Readouts are tokenizer strings. They are associated with the transported residual.",
            "The steer uses the output-embedding row of a top readout token, scaled by STEER_RESIDUAL_SCALE times the residual norm.",
        ],
        "reading_limit": "A readout supports an association between an intermediate residual and a vocabulary item. It leaves any step-by-step reasoning trace unmeasured.",
    },
    "patching": {
        "baseline": "Dumas et al. (2025), Separating Tongue from Thought",
        "code": [
            "src/multilingual_mechinterp/patching/collect.py",
            "src/multilingual_mechinterp/patching/intervene.py",
            "src/multilingual_mechinterp/patching/patch.py",
        ],
        "component": "Residual stream at one layer and the last real token. Source prompt is English. Target prompt is the Persian parallel.",
        "procedure": "Cache the English residual, overwrite that layer in the Persian forward, and record the change in MCQ logit margin for the correct option letter.",
        "departures": [
            "window=1 replaces one layer. The package default replaces the span from the start layer through the final layer.",
            "The score is the correct-letter margin on the answer options, together with the predicted letter.",
            "A layer effect is the effect of that swap. It is the causal reference for the comparison, and it is silent about the rest of the circuit.",
        ],
    },
}

prior_root = ROOT / "output" / "gemma_9B"
prior_files = []
if prior_root.exists():
    prior_files = sorted(
        str(path.relative_to(ROOT))
        for path in prior_root.rglob("*")
        if path.is_file() and path.suffix in {".json", ".csv", ".png", ".md"}
    )
prior_note = {
    "path": str(prior_root) if prior_root.exists() else None,
    "n_files": len(prior_files),
    "files": prior_files,
    "use": (
        "Supplementary record of the earlier Gemma pipeline notebook. "
        "Placement figures and numbers in the report should come from output/placement_1, "
        "because this run uses single-layer patching and records interventions."
    ),
}
dump("method_notes", method_notes)
dump("prior_gemma_artefacts", prior_note)
print(f"Prior Gemma artefacts indexed: {len(prior_files)}")


## Stage 5 — Shared prompt set

The plan schedules the dataset in week 5. It is built here first so SAE, JLens, and patching all see the same items.

Every selected row is a curated multiple-choice question with English, Persian, Bengali, and Urdu text. Categories:

- **semantically matched** and **controlled causal pair**: English prompt as source, Persian prompt as target, same correct letter.
- **multilingual variant**: Bengali and Urdu text for the same item, scored in the behavioural baseline.
- **intermediate inference**: the English question contains one of `symbol`, `refer`, `gesture`, `custom`, `relationship`. Other items stay factual lookups.

The parent file is the reviewed pool. This cell subsets it. It does not ask a model to invent new questions. To change the sample, set `ITEM_IDS` and run from this cell downward.


In [ ]:
items_all = load_culture_questions(culture_json_path())
if ITEM_IDS:
    by_id = {item.question_number: item for item in items_all}
    missing = [qid for qid in ITEM_IDS if qid not in by_id]
    if missing:
        raise KeyError(f"Question ids not in the culture file: {missing}")
    items = [by_id[qid] for qid in ITEM_IDS]
else:
    items = items_all[:N_ITEMS]

if not items:
    raise RuntimeError(f"No culture items at {culture_json_path()}")

PAIRS = []
for item in items:
    source, target, source_answer, target_answer = item.parallel_prompts("english", "persian")
    letter = chr(ord("A") + int(item.correct_index))
    question = item.text("english")
    categories = ["semantically_matched", "multilingual_variant", "controlled_causal_pair"]
    if any(marker in question.lower() for marker in INFERENCE_MARKERS):
        categories.append("intermediate_inference")
    others = {}
    for lang in ("bengali", "urdu"):
        others[lang] = {
            "prompt": item.mcq_prompt(lang),
            "answer_text": item.correct_answer(lang),
        }
    PAIRS.append(
        {
            "id": str(item.question_number),
            "difficulty": item.difficulty,
            "categories": categories,
            "correct_letter": letter,
            "correct_index": int(item.correct_index),
            "source_lang": "english",
            "target_lang": "persian",
            "question_en": question,
            "source_prompt": source,
            "target_prompt": target,
            "source_answer": source_answer,
            "target_answer": target_answer,
            "other_languages": others,
        }
    )

n_inference = sum("intermediate_inference" in pair["categories"] for pair in PAIRS)
prompt_record = {
    "n_pool": len(items_all),
    "n_selected": len(PAIRS),
    "selection": "ITEM_IDS" if ITEM_IDS else f"first {N_ITEMS} items in file order",
    "size_note": (
        f"Selected {len(PAIRS)} of {len(items_all)} curated questions. "
        "The placement plan's 30–100 range can be met by raising N_ITEMS. "
        "This default keeps Jacobian fitting and the layer sweep inside a single report run."
    ),
    "n_intermediate_inference": n_inference,
    "inference_rule": list(INFERENCE_MARKERS),
    "items": PAIRS,
}
dump("prompt_set", prompt_record)
print(f"Selected {len(PAIRS)} items; {n_inference} match the inference markers.")
for pair in PAIRS:
    print(f"  Q{pair['id']} [{pair['correct_letter']}] {pair['question_en'][:88]}")


## Baseline behaviour

Score the correct option letter on English, Persian, Bengali, and Urdu before any intervention. Later methods try to explain this behaviour. The recorded fields are the predicted letter, whether it matches the key, P(correct) among A–D, and the logit margin of the correct letter over the strongest alternative.


In [ ]:
BASELINE_ROWS = []
try:
    with timed("baseline"):
        for pair in PAIRS:
            row = {
                "id": pair["id"],
                "correct_letter": pair["correct_letter"],
                "categories": pair["categories"],
                "english": score_prompt(pair["source_prompt"], pair["correct_letter"]),
                "persian": score_prompt(pair["target_prompt"], pair["correct_letter"]),
            }
            for lang, block in pair["other_languages"].items():
                row[lang] = score_prompt(block["prompt"], pair["correct_letter"])
            BASELINE_ROWS.append(row)
            en, fa = row["english"], row["persian"]
            print(
                f"Q{pair['id']} key={pair['correct_letter']}  "
                f"EN {en['pred_letter']} p={en['p_correct']:.3f}  "
                f"FA {fa['pred_letter']} p={fa['p_correct']:.3f}"
            )
    dump("baseline_behaviour", BASELINE_ROWS)
    STATUS["baseline"] = "ok"
except Exception:
    STATUS["baseline"] = "failed"
    traceback.print_exc()

if BASELINE_ROWS:
    langs = ["english", "persian", "bengali", "urdu"]
    x = np.arange(len(BASELINE_ROWS))
    width = 0.18
    fig, ax = plt.subplots(figsize=(10, 4))
    for i, lang in enumerate(langs):
        vals = [row[lang]["p_correct"] for row in BASELINE_ROWS]
        ax.bar(x + (i - 1.5) * width, vals, width=width, label=lang)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Q{row['id']}" for row in BASELINE_ROWS])
    ax.set_ylim(0, 1)
    ax.set_ylabel("P(correct letter | A–D)")
    ax.set_title("Baseline multiple-choice confidence")
    ax.legend(ncol=4, fontsize=9)
    savefig(fig, "01_baseline_p_correct.png")
empty_cache()


## Stage 2 — Sparse autoencoder

Training uses residual tokens from the English and Persian prompts at one layer. The sweep picks the L1 coefficient whose mean L0 is smallest among runs within 0.05 FVU of the best reconstruction. That checkpoint is saved to `output/placement_1/sae/chosen_tied_sae.pt`.

Feature charts show which latent units fire on each prompt. Overlap between English and Persian is a consistency check. It is a weak guide to meaning until you inspect contexts or ablate the unit.


In [ ]:
if not RUN_SAE:
    print("RUN_SAE is False — skip training.")
    STATUS["sae_train"] = "skipped"
else:
    try:
        with timed("sae_train"):
            if SAE_LAYER is not None:
                SAE_LAYER_USED = int(SAE_LAYER)
            else:
                SAE_LAYER_USED = int(cfg.get("sae", {}).get("layer") or (model.n_layers // 2))
            texts = [pair["source_prompt"] for pair in PAIRS] + [pair["target_prompt"] for pair in PAIRS]
            acts = extract_residual_activations(
                model,
                texts,
                layer=SAE_LAYER_USED,
                max_length=MAX_LENGTH,
                batch_size=1,
            )
            n_raw = int(acts.shape[0])
            if n_raw > SAE_MAX_TOKENS:
                generator = torch.Generator().manual_seed(SEED)
                keep = torch.randperm(n_raw, generator=generator)[:SAE_MAX_TOKENS]
                acts = acts[keep]
            device = SAE_DEVICE or str(model.device)
            sweep_rows = sweep_l1(
                acts,
                SAE_ALPHAS,
                ratio=SAE_RATIO,
                output_dir=SAE_DIR / "l1_sweep",
                n_epochs=SAE_EPOCHS,
                batch_size=min(SAE_BATCH, max(1, int(acts.shape[0]))),
                device=device,
                save_checkpoints=False,
            )
            best_fvu = min(row["fvu"] for row in sweep_rows)
            pool = [row for row in sweep_rows if row["fvu"] <= best_fvu + 0.05]
            chosen = min(pool, key=lambda row: row["mean_l0"])
            SAE = chosen["sae"]
            SAE.save(SAE_DIR / "chosen_tied_sae.pt")
            metrics = [{k: v for k, v in row.items() if k != "sae"} for row in sweep_rows]
            dump(
                "sae_sweep",
                {
                    "layer": SAE_LAYER_USED,
                    "n_tokens_raw": n_raw,
                    "n_tokens_train": int(acts.shape[0]),
                    "d_model": int(acts.shape[-1]),
                    "ratio": SAE_RATIO,
                    "chosen_alpha": chosen["alpha"],
                    "chosen_fvu": chosen["fvu"],
                    "chosen_mean_l0": chosen["mean_l0"],
                    "selection_rule": "lowest mean_l0 among runs within 0.05 FVU of the best FVU",
                    "rows": metrics,
                },
            )
            fig, ax = plt.subplots()
            ax.scatter([row["mean_l0"] for row in metrics], [row["fvu"] for row in metrics], s=60)
            for row in metrics:
                ax.annotate(f"{row['alpha']:.1e}", (row["mean_l0"], row["fvu"]), fontsize=8)
            ax.set_xlabel("Mean L0")
            ax.set_ylabel("FVU")
            ax.set_title(f"Tied SAE at layer {SAE_LAYER_USED}")
            savefig(fig, "02_sae_pareto.png")
        STATUS["sae_train"] = "ok"
    except Exception:
        STATUS["sae_train"] = "failed"
        traceback.print_exc()
    empty_cache()


In [ ]:
SAE_FEATURE_ROWS = []
if STATUS.get("sae_train") != "ok":
    print("SAE training did not finish — skip feature analysis.")
else:
    try:
        with timed("sae_features"):
            english = analyze_features(
                model,
                [pair["source_prompt"] for pair in PAIRS],
                SAE_LAYER_USED,
                sae=SAE,
                top_k=SAE_TOP_K,
            )
            persian = analyze_features(
                model,
                [pair["target_prompt"] for pair in PAIRS],
                SAE_LAYER_USED,
                sae=SAE,
                top_k=SAE_TOP_K,
            )
            for pair, en, fa in zip(PAIRS, english, persian):
                en_ids = [item["feature_id"] for item in en.top_features]
                fa_ids = [item["feature_id"] for item in fa.top_features]
                SAE_FEATURE_ROWS.append(
                    {
                        "id": pair["id"],
                        "english": en.top_features,
                        "persian": fa.top_features,
                        "shared_ids": sorted(set(en_ids) & set(fa_ids)),
                        "jaccard_top_k": jaccard(en_ids, fa_ids),
                    }
                )
                print(
                    f"Q{pair['id']} Jaccard={SAE_FEATURE_ROWS[-1]['jaccard_top_k']:.2f} "
                    f"shared={SAE_FEATURE_ROWS[-1]['shared_ids']}"
                )
            showcase = PAIRS[SHOWCASE_INDEX]
            extra = {}
            for lang in ("bengali", "urdu"):
                block = showcase["other_languages"][lang]
                result = analyze_features(
                    model, block["prompt"], SAE_LAYER_USED, sae=SAE, top_k=SAE_TOP_K
                )[0]
                extra[lang] = [item["feature_id"] for item in result.top_features]
            dump(
                "sae_features",
                {
                    "layer": SAE_LAYER_USED,
                    "top_k": SAE_TOP_K,
                    "items": SAE_FEATURE_ROWS,
                    "showcase_other_language_ids": extra,
                },
            )
            show = SAE_FEATURE_ROWS[SHOWCASE_INDEX]
            fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
            for ax, lang in zip(axes, ("english", "persian")):
                feats = show[lang][:8]
                ax.barh(
                    [str(item["feature_id"]) for item in feats][::-1],
                    [item["activation"] for item in feats][::-1],
                )
                ax.set_title(f"Q{show['id']} {lang}")
                ax.set_xlabel("Feature activation")
            fig.suptitle(f"Strongest SAE features at layer {SAE_LAYER_USED}")
            savefig(fig, "03_sae_top_features_showcase.png")
            fig, ax = plt.subplots(figsize=(8, 3.5))
            ax.bar(
                [f"Q{row['id']}" for row in SAE_FEATURE_ROWS],
                [row["jaccard_top_k"] for row in SAE_FEATURE_ROWS],
            )
            ax.set_ylim(0, 1)
            ax.set_ylabel(f"Jaccard of top-{SAE_TOP_K} features")
            ax.set_title("English–Persian feature overlap")
            savefig(fig, "04_sae_feature_jaccard.png")
        STATUS["sae_features"] = "ok"
    except Exception:
        STATUS["sae_features"] = "failed"
        traceback.print_exc()
    empty_cache()


## SAE feature intervention

On the showcase Persian prompt, take the strongest feature at the SAE layer, zero it, and separately increase it. Decode back to a residual, write that residual into the same layer, and rescore the MCQ. The clean reconstruction is a control: if reconstruction alone moves the answer, later ablation effects are mixed with SAE error.


In [ ]:
SAE_INTERVENTION = None
if not RUN_INTERVENTIONS or STATUS.get("sae_features") != "ok":
    print("Skip SAE intervention.")
    STATUS["sae_intervention"] = "skipped"
else:
    try:
        with timed("sae_intervention"):
            pair = PAIRS[SHOWCASE_INDEX]
            prompt = pair["target_prompt"]
            letter = pair["correct_letter"]
            residual, pos = last_residual(prompt, SAE_LAYER_USED)
            sae = SAE.to(model.device).eval()
            residual_d = residual.to(model.device)
            with torch.no_grad():
                code, recon = sae(residual_d)
            feature_id = int(torch.argmax(code).item())
            strength = float(code[feature_id].abs().clamp_min(1.0).item())
            ablated = ablate_features(residual_d, sae, [feature_id]).detach().float().cpu()
            steered = steer_features(
                residual_d, sae, [feature_id], strength=strength
            ).detach().float().cpu()
            conditions = {
                "clean": score_prompt(prompt, letter),
                "reconstruction": score_logits(
                    logits_with_residual(prompt, SAE_LAYER_USED, recon.detach().float().cpu(), pos),
                    letter,
                ),
                "ablate": score_logits(
                    logits_with_residual(prompt, SAE_LAYER_USED, ablated, pos),
                    letter,
                ),
                "steer": score_logits(
                    logits_with_residual(prompt, SAE_LAYER_USED, steered, pos),
                    letter,
                ),
            }
            SAE_INTERVENTION = {
                "id": pair["id"],
                "prompt_lang": "persian",
                "layer": SAE_LAYER_USED,
                "feature_id": feature_id,
                "feature_activation": float(code[feature_id].item()),
                "steer_strength": strength,
                "conditions": conditions,
            }
            dump("sae_intervention", SAE_INTERVENTION)
            for name, score in conditions.items():
                print(
                    f"{name:16} pred={score['pred_letter']}  "
                    f"p={score['p_correct']:.3f}  margin={score['margin']:.3f}"
                )
        STATUS["sae_intervention"] = "ok"
    except Exception:
        STATUS["sae_intervention"] = "failed"
        traceback.print_exc()
    empty_cache()


## Stage 3 — Jacobian Lens

Fit Jacobians on the first `JLENS_FIT_PROMPTS` English prompts, using the last `JLENS_N_SOURCE_LAYERS` blocks before the final layer. `run_jlens` then ranks tokens at those layers for every selected English and Persian prompt.

The letter-logit plot tracks the correct option letter across depth. A rising curve means that letter is more strongly associated with the transported residual in later layers. Token lists are saved in JSON so Persian strings stay intact even when the figure axis cannot draw them.

`force_bos=False` keeps the tokenizer on the settings used for the baseline and the patching runs.


In [ ]:
JLENS_ROWS = []
LENS = None
JLENS_LAYERS = []
if not RUN_JLENS:
    print("RUN_JLENS is False — skip.")
    STATUS["jlens"] = "skipped"
else:
    try:
        with timed("jlens"):
            lens_model = from_hf(model.model, model.tokenizer, force_bos=False)
            n_src = min(JLENS_N_SOURCE_LAYERS, model.n_layers - 1)
            JLENS_LAYERS = list(range(model.n_layers - 1 - n_src, model.n_layers - 1))
            fit_prompts = [pair["source_prompt"] for pair in PAIRS[:JLENS_FIT_PROMPTS]]
            lengths = []
            for text in fit_prompts:
                ids = model.tokenizer(text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
                lengths.append(len(ids))
            skip = JLENS_SKIP_FIRST
            if min(lengths) <= skip + 1:
                skip = max(0, min(lengths) - 2)
                print(f"skip_first lowered to {skip} (shortest fit prompt has {min(lengths)} tokens)")
            print(f"Fitting layers {JLENS_LAYERS} on {len(fit_prompts)} prompts, dim_batch={JLENS_DIM_BATCH}")
            LENS = fit(
                lens_model,
                fit_prompts,
                source_layers=JLENS_LAYERS,
                dim_batch=JLENS_DIM_BATCH,
                max_seq_len=MAX_LENGTH,
                skip_first=skip,
                checkpoint_path=None,
            )
            opt_ids = option_token_ids(model.tokenizer)
            for pair in PAIRS:
                letter_id = opt_ids[pair["correct_letter"]]
                per_lang = {}
                for lang, prompt in (("english", pair["source_prompt"]), ("persian", pair["target_prompt"])):
                    result = run_jlens(
                        lens_model,
                        prompt,
                        lens=LENS,
                        layers=JLENS_LAYERS,
                        top_k=JLENS_TOP_K,
                        use_jacobian=True,
                        max_seq_len=MAX_LENGTH,
                    )
                    per_lang[lang] = {
                        "letter_logit": [float(result.logits[layer][letter_id]) for layer in JLENS_LAYERS],
                        "top": [
                            [
                                {
                                    "token": item["token"],
                                    "token_id": item["token_id"],
                                    "logit": item["logit"],
                                }
                                for item in result.top_tokens[layer]
                            ]
                            for layer in JLENS_LAYERS
                        ],
                    }
                en_top = [item["token"].strip() for item in per_lang["english"]["top"][-1]]
                fa_top = [item["token"].strip() for item in per_lang["persian"]["top"][-1]]
                JLENS_ROWS.append(
                    {
                        "id": pair["id"],
                        "correct_letter": pair["correct_letter"],
                        "layers": list(JLENS_LAYERS),
                        "late_top5_jaccard": jaccard(en_top[:5], fa_top[:5]),
                        "english": per_lang["english"],
                        "persian": per_lang["persian"],
                    }
                )
                print(f"Q{pair['id']} late top-5 Jaccard={JLENS_ROWS[-1]['late_top5_jaccard']:.2f}")
            showcase_prompt = PAIRS[SHOWCASE_INDEX]["source_prompt"]
            logit_only = run_jlens(
                lens_model,
                showcase_prompt,
                lens=LENS,
                layers=JLENS_LAYERS,
                top_k=5,
                use_jacobian=False,
                max_seq_len=MAX_LENGTH,
            )
            letter_id = opt_ids[PAIRS[SHOWCASE_INDEX]["correct_letter"]]
            logit_curve = [float(logit_only.logits[layer][letter_id]) for layer in JLENS_LAYERS]
            dump(
                "jlens_readouts",
                {
                    "source_layers": list(JLENS_LAYERS),
                    "n_fit_prompts": int(LENS.n_prompts),
                    "skip_first": skip,
                    "fit_prompt_ids": [pair["id"] for pair in PAIRS[:JLENS_FIT_PROMPTS]],
                    "items": JLENS_ROWS,
                    "showcase_logit_lens_letter_logit": logit_curve,
                },
            )
            fig, ax = plt.subplots()
            show = JLENS_ROWS[SHOWCASE_INDEX]
            ax.plot(JLENS_LAYERS, show["english"]["letter_logit"], marker="o", label=f"Q{show['id']} EN Jacobian")
            ax.plot(JLENS_LAYERS, show["persian"]["letter_logit"], marker="o", label=f"Q{show['id']} FA Jacobian")
            ax.plot(JLENS_LAYERS, logit_curve, marker="s", label=f"Q{show['id']} EN logit lens")
            ax.set_xlabel("Source layer")
            ax.set_ylabel(f"Logit of correct letter {show['correct_letter']}")
            ax.set_title("Correct-letter readout across fitted layers")
            ax.legend(fontsize=8)
            savefig(fig, "05_jlens_letter_logit.png")
            print("Showcase late-layer tokens (English):")
            for item in show["english"]["top"][-1][:8]:
                print(f"  {item['logit']:8.2f}  {item['token']!r}")
            print("Showcase late-layer tokens (Persian):")
            for item in show["persian"]["top"][-1][:8]:
                print(f"  {item['logit']:8.2f}  {item['token']!r}")
        STATUS["jlens"] = "ok"
    except Exception:
        STATUS["jlens"] = "failed"
        traceback.print_exc()
    empty_cache()


## Jacobian Lens concept-direction intervention

Take the top token at the latest fitted layer on the showcase Persian prompt. Add and subtract its output-embedding direction at that layer (`intervene_jlens`), with scale `STEER_RESIDUAL_SCALE * ||residual||`, then rescore the MCQ. This tests whether pushing the residual toward a lens-ranked vocabulary direction moves the answer.


In [ ]:
JLENS_INTERVENTION = None
if not RUN_INTERVENTIONS or STATUS.get("jlens") != "ok":
    print("Skip JLens intervention.")
    STATUS["jlens_intervention"] = "skipped"
else:
    try:
        with timed("jlens_intervention"):
            pair = PAIRS[SHOWCASE_INDEX]
            prompt = pair["target_prompt"]
            letter = pair["correct_letter"]
            latest = JLENS_LAYERS[-1]
            top = JLENS_ROWS[SHOWCASE_INDEX]["persian"]["top"][-1][0]
            weight = model.model.get_output_embeddings().weight[top["token_id"]].detach().float().cpu()
            direction = weight / weight.norm().clamp_min(1e-8)
            residual, pos = last_residual(prompt, latest)
            scale = STEER_RESIDUAL_SCALE * float(residual.norm())
            strengthened = intervene_jlens(residual, direction=direction, scale=scale)
            suppressed = intervene_jlens(residual, direction=direction, scale=-scale)
            conditions = {
                "clean": score_prompt(prompt, letter),
                "strengthen": score_logits(logits_with_residual(prompt, latest, strengthened, pos), letter),
                "suppress": score_logits(logits_with_residual(prompt, latest, suppressed, pos), letter),
            }
            JLENS_INTERVENTION = {
                "id": pair["id"],
                "prompt_lang": "persian",
                "layer": latest,
                "token": top["token"],
                "token_id": top["token_id"],
                "token_logit": top["logit"],
                "scale": scale,
                "conditions": conditions,
            }
            dump("jlens_intervention", JLENS_INTERVENTION)
            for name, score in conditions.items():
                print(
                    f"{name:12} pred={score['pred_letter']}  "
                    f"p={score['p_correct']:.3f}  margin={score['margin']:.3f}"
                )
        STATUS["jlens_intervention"] = "ok"
    except Exception:
        STATUS["jlens_intervention"] = "failed"
        traceback.print_exc()
    empty_cache()


## Stage 4 — Activation patching

For each item, cache the English last-token residual and write it into the Persian run at one layer (`window=1`). The curve is the change in correct-letter logit margin.

On the layer with the largest absolute margin change, the same swap is run once more. A second hook reads the residual at that position and compares it with the English vector, so the record shows that the replacement happened. The predicted letter after that swap is stored next to the clean Persian prediction.


In [ ]:
PATCH_ROWS = []
if not RUN_PATCHING:
    print("RUN_PATCHING is False — skip.")
    STATUS["patching"] = "skipped"
else:
    try:
        with timed("patching"):
            blocks = get_blocks(model.model)
            dtype = next(model.model.parameters()).dtype
            for pair in PAIRS:
                result = run_patching(
                    model,
                    source_prompt=pair["source_prompt"],
                    target_prompt=pair["target_prompt"],
                    answer=pair["correct_letter"],
                    window=PATCH_WINDOW,
                    metric="margin",
                )
                layers = list(result.layers)
                deltas = [float(result.scores[layer]) for layer in layers]
                abs_i = max(range(len(deltas)), key=lambda i: abs(deltas[i]))
                best_layer = int(layers[abs_i])
                source_acts = collect_activations(model, [pair["source_prompt"]], pos_idx=-1)
                tok, pos = encode_prompt(pair["target_prompt"])
                values = source_acts[best_layer, 0].to(device=model.device, dtype=dtype).unsqueeze(0)
                positions = torch.tensor([pos], device=model.device)
                seen = {}

                def _checker(_module, _inp, out, _pos=pos):
                    hidden = out[0] if isinstance(out, tuple) else out
                    seen["residual"] = hidden[0, _pos].detach().float().cpu()

                with torch.no_grad(), patch_residuals(model.model, {best_layer: values}, positions):
                    handle = blocks[best_layer].register_forward_hook(_checker)
                    try:
                        out = model.model(**tok)
                    finally:
                        handle.remove()
                patched = score_logits(out.logits[0, pos].detach().float().cpu(), pair["correct_letter"])
                source_vec = source_acts[best_layer, 0].float()
                if "residual" not in seen:
                    max_abs_diff = float("inf")
                else:
                    roundtrip = source_vec.to(dtype=dtype).float()
                    max_abs_diff = float((seen["residual"] - roundtrip).abs().max().item())
                clean_row = next((row for row in BASELINE_ROWS if row["id"] == pair["id"]), None)
                clean = {"pred_letter": None} if clean_row is None else clean_row["persian"]
                PATCH_ROWS.append(
                    {
                        "id": pair["id"],
                        "correct_letter": pair["correct_letter"],
                        "window": PATCH_WINDOW,
                        "baseline_margin": float(result.baseline_score),
                        "layers": layers,
                        "delta_margin": deltas,
                        "max_abs_layer": best_layer,
                        "max_abs_delta": deltas[abs_i],
                        "clean_pred": clean["pred_letter"],
                        "patched_pred": patched["pred_letter"],
                        "patched_margin": patched["margin"],
                        "patched_p_correct": patched["p_correct"],
                        "letter_changed": clean["pred_letter"] is not None and patched["pred_letter"] != clean["pred_letter"],
                        "replacement_max_abs_diff": max_abs_diff,
                        "replacement_ok": max_abs_diff < 0.05,
                    }
                )
                print(
                    f"Q{pair['id']} max |Δmargin|={abs(deltas[abs_i]):.3f} at layer {best_layer}  "
                    f"pred {clean['pred_letter']}→{patched['pred_letter']}  "
                    f"replace_diff={max_abs_diff:.4f}"
                )
            dump("patching", PATCH_ROWS)
            fig, ax = plt.subplots(figsize=(9, 4))
            for row in PATCH_ROWS:
                ax.plot(row["layers"], row["delta_margin"], label=f"Q{row['id']}", linewidth=1.2)
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set_xlabel("Layer whose residual was replaced")
            ax.set_ylabel("Δ margin (patched − clean Persian)")
            ax.set_title("Single-layer English → Persian activation patch")
            ax.legend(ncol=4, fontsize=8)
            savefig(fig, "06_patch_delta_margin.png")
        STATUS["patching"] = "ok"
    except Exception:
        STATUS["patching"] = "failed"
        traceback.print_exc()
    empty_cache()


## Stage 6 — Same prompts, three methods

Each item row puts the SAE overlap, the late-layer token overlap, and the largest single-layer patch effect side by side. The six-criterion table answers the comparison the plan asks for: representation, layers, interpretability, causal evidence, computation, and multilingual usefulness.

Layer numbers that coincide are reported as a count. A shared depth is a prompt for a follow-up test. The patch effect remains the causal measurement; the SAE and the lens describe the residual at their own sites.


In [ ]:
by_id_sae = {row["id"]: row for row in SAE_FEATURE_ROWS}
by_id_jl = {row["id"]: row for row in JLENS_ROWS}
by_id_patch = {row["id"]: row for row in PATCH_ROWS}
by_id_base = {row["id"]: row for row in BASELINE_ROWS}

comparison_rows = []
for pair in PAIRS:
    qid = pair["id"]
    base = by_id_base.get(qid, {})
    sae_row = by_id_sae.get(qid)
    jl_row = by_id_jl.get(qid)
    patch_row = by_id_patch.get(qid)
    jlens_peak = None
    if jl_row is not None:
        curve = jl_row["persian"]["letter_logit"]
        jlens_peak = int(jl_row["layers"][int(np.argmax(curve))])
    comparison_rows.append(
        {
            "id": qid,
            "correct_letter": pair["correct_letter"],
            "categories": pair["categories"],
            "en_correct": None if not base else base["english"]["correct"],
            "fa_correct": None if not base else base["persian"]["correct"],
            "en_p": None if not base else base["english"]["p_correct"],
            "fa_p": None if not base else base["persian"]["p_correct"],
            "sae_jaccard": None if sae_row is None else sae_row["jaccard_top_k"],
            "sae_shared": [] if sae_row is None else sae_row["shared_ids"],
            "jlens_late_jaccard": None if jl_row is None else jl_row["late_top5_jaccard"],
            "jlens_peak_layer_fa": jlens_peak,
            "patch_max_abs_layer": None if patch_row is None else patch_row["max_abs_layer"],
            "patch_max_abs_delta": None if patch_row is None else patch_row["max_abs_delta"],
            "patch_letter_changed": None if patch_row is None else patch_row["letter_changed"],
            "replacement_ok": None if patch_row is None else patch_row["replacement_ok"],
        }
    )

sae_in_patch = []
sae_in_jlens = []
for row in comparison_rows:
    if SAE_LAYER_USED is not None and row["patch_max_abs_layer"] is not None:
        sae_in_patch.append(int(row["patch_max_abs_layer"]) == int(SAE_LAYER_USED))
    if SAE_LAYER_USED is not None and JLENS_LAYERS:
        sae_in_jlens.append(int(SAE_LAYER_USED) in set(JLENS_LAYERS))


def _mean(values):
    vals = [v for v in values if v is not None]
    return None if not vals else float(sum(vals) / len(vals))


summary_stats = {
    "n_items": len(comparison_rows),
    "mean_sae_jaccard": _mean([row["sae_jaccard"] for row in comparison_rows]),
    "mean_jlens_late_jaccard": _mean([row["jlens_late_jaccard"] for row in comparison_rows]),
    "mean_abs_patch_delta": _mean(
        [None if row["patch_max_abs_delta"] is None else abs(row["patch_max_abs_delta"]) for row in comparison_rows]
    ),
    "n_letter_changes": sum(bool(row["patch_letter_changed"]) for row in comparison_rows),
    "sae_layer": SAE_LAYER_USED,
    "n_patch_peak_equals_sae_layer": sum(sae_in_patch),
    "n_patch_compared": len(sae_in_patch),
    "sae_layer_inside_jlens_window": bool(sae_in_jlens[0]) if sae_in_jlens else None,
    "jlens_layers": list(JLENS_LAYERS),
    "stage_times_s": STAGE_TIMES,
    "status": STATUS,
}

criteria = [
    {
        "criterion": "Internal representation",
        "sae": "Sparse feature activations on one residual layer. Overlap is the Jaccard of the top-k feature ids.",
        "jlens": "Vocabulary tokens associated with Jacobian-transported residuals, plus the logit of the correct option letter.",
        "patching": "Change in the Persian MCQ margin after the English residual replaces one layer.",
    },
    {
        "criterion": "Important layers",
        "sae": f"Feature analysis uses layer {SAE_LAYER_USED}.",
        "jlens": f"Readouts use source layers {list(JLENS_LAYERS)}. Persian letter-logit peaks are in the item table.",
        "patching": (
            f"The largest |Δmargin| matches the SAE layer on "
            f"{summary_stats['n_patch_peak_equals_sae_layer']} of {summary_stats['n_patch_compared']} items."
        ),
    },
    {
        "criterion": "Interpretability",
        "sae": "Feature ids are numeric. Shared ids are consistent activations, and they still need a context or an ablation before a name is justified.",
        "jlens": "Top tokens are readable strings from the model tokenizer. Late-layer overlap with the English prompt is in the item table.",
        "patching": "The result is a signed margin change and, on the strongest layer, whether the predicted letter moved.",
    },
    {
        "criterion": "Causal evidence",
        "sae": "Ablation and steering on the showcase item are causal for that one feature write-back. The ranking itself is descriptive.",
        "jlens": "The direction steer on the showcase item is causal for that vector add. Token ranks themselves are descriptive.",
        "patching": "Single-layer residual replacement is the main causal measurement in this placement.",
    },
    {
        "criterion": "Computation",
        "sae": f"Training plus feature reads: see stage_times.s ({json.dumps(STAGE_TIMES)}).",
        "jlens": "Fitting dominates. Cost scales with fit prompts, source layers, and d_model / JLENS_DIM_BATCH.",
        "patching": "One target forward per layer per item, plus one verification forward on the strongest layer.",
    },
    {
        "criterion": "Multilingual suitability",
        "sae": f"Mean EN–FA top-k Jaccard = {summary_stats['mean_sae_jaccard']}. Bengali and Urdu ids for the showcase item are in sae_features.json.",
        "jlens": f"Mean late-layer top-5 token Jaccard = {summary_stats['mean_jlens_late_jaccard']}.",
        "patching": (
            f"Mean |Δmargin| = {summary_stats['mean_abs_patch_delta']}; "
            f"predicted letter changed on {summary_stats['n_letter_changes']} items."
        ),
    },
]

dump("comparison_items", comparison_rows)
dump("comparison_summary", summary_stats)
dump("comparison_criteria", criteria)

csv_path = REC / "comparison_items.csv"
if comparison_rows:
    fieldnames = list(comparison_rows[0].keys())
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in comparison_rows:
            flat = {key: json.dumps(val) if isinstance(val, (list, dict)) else val for key, val in row.items()}
            writer.writerow(flat)
    print("wrote", csv_path)

if comparison_rows and any(row["sae_jaccard"] is not None for row in comparison_rows):
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
    labels = [f"Q{row['id']}" for row in comparison_rows]
    series = [
        ("SAE Jaccard", [0 if row["sae_jaccard"] is None else row["sae_jaccard"] for row in comparison_rows], (0, 1)),
        (
            "JLens late Jaccard",
            [0 if row["jlens_late_jaccard"] is None else row["jlens_late_jaccard"] for row in comparison_rows],
            (0, 1),
        ),
        (
            "Max |Δmargin|",
            [0 if row["patch_max_abs_delta"] is None else abs(row["patch_max_abs_delta"]) for row in comparison_rows],
            None,
        ),
    ]
    for ax, (title, values, ylim) in zip(axes, series):
        ax.bar(labels, values)
        ax.set_title(title)
        if ylim:
            ax.set_ylim(*ylim)
        ax.tick_params(axis="x", labelrotation=45)
    fig.tight_layout()
    savefig(fig, "07_method_summary.png")

print(json.dumps(summary_stats, indent=2, default=str))


## Stage 7 — Research record and report notes

The last cell writes `output/placement_1/records/report_notes.md` and a checklist. Use the notes as the spine of the results and comparison sections. Quote numbers from the JSON files in the same folder, and use the PNGs in `output/placement_1/figures/` as the report figures.

Examiner names and the oral presentation stay with your mentor. This notebook records the experiments those submissions need.


In [ ]:
if "criteria" not in globals():
    criteria_path = REC / "comparison_criteria.json"
    criteria = json.loads(criteria_path.read_text(encoding="utf-8")) if criteria_path.exists() else []

lines = []
lines.append("# Placement 1 run notes")
lines.append("")
lines.append(f"Generated {datetime.now(timezone.utc).isoformat()}.")
lines.append("")
lines.append("## Setup")
lines.append("")
lines.append(f"- Model: `{model.name}`")
lines.append(f"- Layers: {model.n_layers}")
lines.append(f"- Device: {model.device}")
lines.append(f"- Package: {getattr(multilingual_mechinterp, '__version__', 'unknown')}")
lines.append(f"- Git: {setup_record.get('git_commit')}")
lines.append(f"- Items: {len(PAIRS)} of the curated 200 (see `prompt_set.json`)")
lines.append(f"- Stage status: `{json.dumps(STATUS)}`")
lines.append(f"- Stage seconds: `{json.dumps(STAGE_TIMES)}`")
lines.append("")
lines.append("## Baseline")
lines.append("")
if BASELINE_ROWS:
    en_acc = sum(row["english"]["correct"] for row in BASELINE_ROWS) / len(BASELINE_ROWS)
    fa_acc = sum(row["persian"]["correct"] for row in BASELINE_ROWS) / len(BASELINE_ROWS)
    lines.append(f"- English letter accuracy: {en_acc:.2f}")
    lines.append(f"- Persian letter accuracy: {fa_acc:.2f}")
    lines.append("- Per-item probabilities: `baseline_behaviour.json` and `figures/01_baseline_p_correct.png`.")
else:
    lines.append("- Baseline scores were not saved.")
lines.append("")
lines.append("## SAE")
lines.append("")
lines.append(f"- Layer: {SAE_LAYER_USED}")
lines.append(f"- Status: {STATUS.get('sae_train')} / features {STATUS.get('sae_features')} / intervention {STATUS.get('sae_intervention')}")
if SAE_INTERVENTION:
    cond = SAE_INTERVENTION["conditions"]
    lines.append(
        f"- Showcase Q{SAE_INTERVENTION['id']} Persian, feature {SAE_INTERVENTION['feature_id']}: "
        f"clean margin {cond['clean']['margin']:.3f}, "
        f"reconstruction {cond['reconstruction']['margin']:.3f}, "
        f"ablate {cond['ablate']['margin']:.3f}, "
        f"steer {cond['steer']['margin']:.3f}."
    )
lines.append("- Files: `sae_sweep.json`, `sae_features.json`, `sae_intervention.json`, figures 02–04.")
lines.append("")
lines.append("## Jacobian Lens")
lines.append("")
lines.append(f"- Source layers: {list(JLENS_LAYERS)}")
lines.append(f"- Status: {STATUS.get('jlens')} / intervention {STATUS.get('jlens_intervention')}")
if JLENS_INTERVENTION:
    cond = JLENS_INTERVENTION["conditions"]
    token = JLENS_INTERVENTION["token"]
    lines.append(
        f"- Showcase steer of token {token!r} at layer {JLENS_INTERVENTION['layer']}: "
        f"clean margin {cond['clean']['margin']:.3f}, "
        f"strengthen {cond['strengthen']['margin']:.3f}, "
        f"suppress {cond['suppress']['margin']:.3f}."
    )
lines.append("- Files: `jlens_readouts.json`, `jlens_intervention.json`, `figures/05_jlens_letter_logit.png`.")
lines.append("- Readouts associate a residual with vocabulary items. They are not a trace of step-by-step reasoning.")
lines.append("")
lines.append("## Activation patching")
lines.append("")
lines.append(f"- Window: {PATCH_WINDOW} (one layer replaced at a time).")
lines.append(f"- Status: {STATUS.get('patching')}")
if PATCH_ROWS:
    n_ok = sum(bool(row["replacement_ok"]) for row in PATCH_ROWS)
    lines.append(f"- Residual replacement check passed on {n_ok} of {len(PATCH_ROWS)} items (max abs diff < 0.05).")
    lines.append(f"- Predicted letter changed on {sum(bool(row['letter_changed']) for row in PATCH_ROWS)} items.")
lines.append("- Files: `patching.json`, `figures/06_patch_delta_margin.png`.")
lines.append("")
lines.append("## Comparison")
lines.append("")
lines.append("| Criterion | SAE | JLens | Patching |")
lines.append("| --- | --- | --- | --- |")
for row in criteria:
    lines.append(f"| {row['criterion']} | {row['sae']} | {row['jlens']} | {row['patching']} |")
lines.append("")
lines.append(
    "Where a patch peak, an SAE layer, and a lens peak sit near one another, report that as convergent description. "
    "The shared depth becomes a single mechanism only after a further test that ties the feature, the direction, and the swap together."
)
lines.append("")
lines.append("## Placement 2")
lines.append("")
lines.append(
    "A practical order for a larger multilingual or cultural dataset is: patch single layers to find depths that move the answer, "
    "read SAE features at those depths, and use Jacobian readouts to see which tokens are associated there. "
    "Keep the prompt set reviewed by hand, and keep the score definition identical across methods."
)
lines.append("")
lines.append("## Figures")
lines.append("")
for path in sorted(FIG.glob("*.png")):
    lines.append(f"- `{path.relative_to(ROOT)}`")

notes_path = REC / "report_notes.md"
notes_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("wrote", notes_path)

checks = [
    ("Research question and environment saved", (REC / "experiment_setup.json").exists()),
    ("Method departures documented", (REC / "method_notes.json").exists()),
    ("Prompt set saved with languages and categories", (REC / "prompt_set.json").exists()),
    ("Baseline letter probabilities saved", (REC / "baseline_behaviour.json").exists()),
    ("SAE sweep metrics saved", (REC / "sae_sweep.json").exists()),
    ("SAE feature ranks saved", (REC / "sae_features.json").exists()),
    ("SAE intervention saved or explicitly skipped", STATUS.get("sae_intervention") in {"ok", "skipped"}),
    ("JLens readouts saved", (REC / "jlens_readouts.json").exists()),
    ("JLens intervention saved or explicitly skipped", STATUS.get("jlens_intervention") in {"ok", "skipped"}),
    ("Single-layer patching results saved", (REC / "patching.json").exists()),
    ("Replacement check recorded", bool(PATCH_ROWS) and all("replacement_ok" in row for row in PATCH_ROWS)),
    ("Cross-method table saved", (REC / "comparison_items.csv").exists()),
    ("Six comparison criteria saved", (REC / "comparison_criteria.json").exists()),
    ("Report notes written", notes_path.exists()),
]
checklist_lines = ["# Placement 1 checklist", ""]
for label, ok in checks:
    mark = "x" if ok else " "
    checklist_lines.append(f"- [{mark}] {label}")
checklist_path = REC / "checklist.md"
checklist_path.write_text("\n".join(checklist_lines) + "\n", encoding="utf-8")
print("wrote", checklist_path)
for label, ok in checks:
    print(("OK  " if ok else "MISS") + " " + label)
